# BI-RADS Baseline Evaluation

Establish the performance floor for all subsequent experiments:
1. **Majority-class baseline** — always predict class 2 (expected macro-F1 ≈ 0.07)
2. **Regex + Logistic Regression baseline** — existing hand-crafted features with balanced LR

Both baselines are evaluated with macro-F1 (the competition metric) and logged to MLflow.

In [ ]:
RANDOM_STATE = 42
N_FOLDS = 5
METRIC = "f1_macro"
DATA_PATH = "data/raw/train.csv"
MLFLOW_EXPERIMENT = "birads-baseline"

In [ ]:
import sys
import tempfile
import warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

matplotlib.use("Agg")
warnings.filterwarnings("ignore")

COMPETITION = "spr-2026-mammography-report-classification"

KAGGLE_INPUT = Path("/kaggle/input") / COMPETITION
if KAGGLE_INPUT.exists():
    DATA_PATH = str(KAGGLE_INPUT / "train.csv")
    IS_KAGGLE = True
else:
    IS_KAGGLE = False

def _find_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists():
            return p
    return Path.cwd()

ROOT = _find_project_root()
_data_path = Path(DATA_PATH)
if not _data_path.is_absolute():
    _data_path = ROOT / _data_path
DATA_PATH = str(_data_path)

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

print(f"Project root: {ROOT}")
print(f"Data path: {DATA_PATH}")
print(f"Kaggle env: {IS_KAGGLE}")

## Load Data

In [ ]:
from preprocessing.preprocess import ACHADOS_PATTERNS, extract_features

train_df = pd.read_csv(DATA_PATH)
print(f"Train shape: {train_df.shape}")
print(f"\nTarget distribution:")
print(train_df["target"].value_counts().sort_index())

train_df = extract_features(train_df)

y = train_df["target"].values
n_classes = len(np.unique(y))
class_labels = sorted(np.unique(y))
print(f"\nClasses: {class_labels} ({n_classes} total)")

## 1. Majority-Class Baseline

Always predict the most frequent class (class 2, 87.4% of training data).  
Expected macro-F1 ≈ 1/7 × (class-2 F1) ≈ 0.07 since all other classes get F1 = 0.

In [ ]:
majority_class = int(pd.Series(y).mode().iloc[0])
majority_preds = np.full_like(y, fill_value=majority_class)

majority_f1_macro = f1_score(y, majority_preds, average="macro", zero_division=0)

print(f"Majority class: {majority_class}")
print(f"Majority-class macro-F1: {majority_f1_macro:.4f}")
print()
print("Per-class classification report:")
print(classification_report(y, majority_preds, zero_division=0))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(y, majority_preds, ax=ax, labels=class_labels)
ax.set_title(f"Majority-Class Baseline — macro-F1 = {majority_f1_macro:.4f}")
plt.tight_layout()
plt.show()

## Log Majority-Class Baseline to MLflow

In [ ]:
if not IS_KAGGLE:
    from dotenv import load_dotenv

    load_dotenv(ROOT / ".env", override=True)

    import os

    for token_var in ("AWS_SESSION_TOKEN", "AWS_SECURITY_TOKEN"):
        os.environ.pop(token_var, None)

    mlflow.set_tracking_uri(
        os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
    )
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    majority_report = classification_report(
        y, majority_preds, output_dict=True, zero_division=0
    )

    with mlflow.start_run(run_name="majority-class-baseline"):
        mlflow.set_tag("baseline", "true")
        mlflow.set_tag("feature_set_version", "v0.0")
        mlflow.set_tag("cv_strategy", f"stratified_{N_FOLDS}fold")
        mlflow.set_tag("model_type", "majority_class")

        mlflow.log_metric("cv_f1_macro", majority_f1_macro)

        for label in class_labels:
            label_str = str(label)
            if label_str in majority_report:
                mlflow.log_metric(
                    f"f1_class_{label}", majority_report[label_str]["f1-score"]
                )

        with tempfile.TemporaryDirectory() as tmpdir:
            fig, ax = plt.subplots(figsize=(8, 6))
            ConfusionMatrixDisplay.from_predictions(
                y, majority_preds, ax=ax, labels=class_labels
            )
            ax.set_title("Majority-Class Baseline — Confusion Matrix")
            cm_path = Path(tmpdir) / "confusion_matrix.png"
            fig.savefig(cm_path, dpi=150, bbox_inches="tight")
            plt.close(fig)
            mlflow.log_artifact(str(cm_path), artifact_path="plots")

    print("Majority-class baseline logged to MLflow.")
else:
    print("Kaggle environment detected — skipping MLflow logging.")

## 2. Regex + Logistic Regression Baseline

Uses the existing `extract_features()` from `src/preprocessing/preprocess.py` (12 binary achado features + 1 categorical indicação + 1 ordinal análise comparativa).  
Fits `LogisticRegression(class_weight='balanced')` with stratified 5-fold CV.

In [ ]:
achado_cols = list(ACHADOS_PATTERNS.keys())
feature_cols = ["indicacao_class", "analise_comparativa"] + achado_cols

X = train_df[feature_cols].copy()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", achado_cols),
        (
            "cat",
            Pipeline(
                [("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]
            ),
            ["indicacao_class"],
        ),
        (
            "txt",
            Pipeline(
                [
                    (
                        "ordinal",
                        OrdinalEncoder(
                            handle_unknown="use_encoded_value", unknown_value=-1
                        ),
                    )
                ]
            ),
            ["analise_comparativa"],
        ),
    ]
)

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

lr_model = LogisticRegression(
    class_weight="balanced",
    solver="saga",
    max_iter=1000,
    random_state=RANDOM_STATE,
)

fold_f1_scores = []
fold_reports = []

for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X, y)):
    X_train_fold = X.iloc[train_idx]
    X_val_fold = X.iloc[val_idx]
    y_train_fold = y[train_idx]
    y_val_fold = y[val_idx]

    X_train_proc = preprocessor.fit_transform(X_train_fold)
    X_val_proc = preprocessor.transform(X_val_fold)

    if sparse.issparse(X_train_proc):
        X_train_proc = X_train_proc.toarray()
        X_val_proc = X_val_proc.toarray()

    lr_model.fit(X_train_proc, y_train_fold)
    y_val_pred = lr_model.predict(X_val_proc)

    fold_f1 = f1_score(y_val_fold, y_val_pred, average="macro", zero_division=0)
    fold_f1_scores.append(fold_f1)

    fold_report = classification_report(
        y_val_fold, y_val_pred, output_dict=True, zero_division=0
    )
    fold_reports.append(fold_report)

    print(f"Fold {fold_idx + 1}/{N_FOLDS}: macro-F1 = {fold_f1:.4f}")

mean_f1 = np.mean(fold_f1_scores)
std_f1 = np.std(fold_f1_scores)
print(f"\nMean macro-F1: {mean_f1:.4f} ± {std_f1:.4f}")

In [ ]:
per_class_f1 = {}
for label in class_labels:
    label_str = str(label)
    class_f1_values = [
        r[label_str]["f1-score"] for r in fold_reports if label_str in r
    ]
    per_class_f1[label] = np.mean(class_f1_values)

print("Per-class mean F1 across folds:")
for label, f1_val in per_class_f1.items():
    print(f"  Class {label}: {f1_val:.4f}")

In [ ]:
X_full_proc = preprocessor.fit_transform(X)
if sparse.issparse(X_full_proc):
    X_full_proc = X_full_proc.toarray()

lr_full = LogisticRegression(
    class_weight="balanced",
    solver="saga",
    max_iter=1000,
    random_state=RANDOM_STATE,
)

oof_preds = cross_val_predict(
    lr_full, X_full_proc, y, cv=cv, method="predict"
)

print("\nFull OOF classification report:")
print(classification_report(y, oof_preds, zero_division=0))

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(y, oof_preds, ax=ax, labels=class_labels)
ax.set_title(f"Regex + LR Baseline — OOF macro-F1 = {mean_f1:.4f}")
plt.tight_layout()
plt.show()

## Log Regex + LR Baseline to MLflow

In [ ]:
if not IS_KAGGLE:
    oof_report = classification_report(
        y, oof_preds, output_dict=True, zero_division=0
    )

    with mlflow.start_run(run_name="regex-lr-baseline"):
        mlflow.set_tag("baseline", "true")
        mlflow.set_tag("feature_set_version", "v1.0")
        mlflow.set_tag("cv_strategy", f"stratified_{N_FOLDS}fold")
        mlflow.set_tag("model_type", "logistic_regression")

        mlflow.log_metric("cv_f1_macro", mean_f1)
        mlflow.log_metric("cv_f1_macro_std", std_f1)

        for fold_i, f1_val in enumerate(fold_f1_scores):
            mlflow.log_metric(f"f1_macro_fold_{fold_i}", f1_val)

        for label in class_labels:
            label_str = str(label)
            if label_str in oof_report:
                mlflow.log_metric(
                    f"f1_class_{label}", oof_report[label_str]["f1-score"]
                )
                mlflow.log_metric(
                    f"precision_class_{label}",
                    oof_report[label_str]["precision"],
                )
                mlflow.log_metric(
                    f"recall_class_{label}", oof_report[label_str]["recall"]
                )

        mlflow.log_param("n_folds", N_FOLDS)
        mlflow.log_param("random_state", RANDOM_STATE)
        mlflow.log_param("solver", "saga")
        mlflow.log_param("class_weight", "balanced")
        mlflow.log_param("max_iter", 1000)

        with tempfile.TemporaryDirectory() as tmpdir:
            fig, ax = plt.subplots(figsize=(8, 6))
            ConfusionMatrixDisplay.from_predictions(
                y, oof_preds, ax=ax, labels=class_labels
            )
            ax.set_title("Regex + LR Baseline — OOF Confusion Matrix")
            cm_path = Path(tmpdir) / "confusion_matrix.png"
            fig.savefig(cm_path, dpi=150, bbox_inches="tight")
            plt.close(fig)
            mlflow.log_artifact(str(cm_path), artifact_path="plots")

    print("Regex + LR baseline logged to MLflow.")
else:
    print("Kaggle environment detected — skipping MLflow logging.")

## Summary

In [ ]:
summary = pd.DataFrame(
    {
        "Model": ["Majority-class (always predict 2)", "Regex + Logistic Regression"],
        "Feature Set": ["v0.0 (none)", "v1.0 (regex)"],
        "macro-F1": [majority_f1_macro, mean_f1],
    }
)
print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#e74c3c", "#3498db"]
bars = ax.barh(summary["Model"], summary["macro-F1"], color=colors)
ax.set_xlabel("macro-F1")
ax.set_title("Baseline Comparison — macro-F1")
ax.set_xlim(0, max(summary["macro-F1"]) * 1.3)
for bar, val in zip(bars, summary["macro-F1"]):
    ax.text(
        bar.get_width() + 0.005,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.4f}",
        va="center",
    )
plt.tight_layout()
plt.show()